# StoreDNA - Pipeline stages

Data flows **top to bottom**. This notebook implements **Stage 5** (highlighted).

```
│  1. Source Data          │
│  2. Curate Data          │
│  3. AI Enrichment        │
│  4. Modality Vectors     │
│  5. StoreDNA Builder ◀── Current Step │
│  6. Vector Index         │
│  7. Business Output      │
```

| Stage | Name | Status |
|-------|------|--------|
| 1 | Source Data | Complete |
| 2 | Curate Data | Complete |
| 3 | AI Enrichment | Complete |
| 4 | Modality Vectors | Complete |
| **5** | **StoreDNA Builder** | **Current** |
| 6 | Vector Index | Next |


# Retail Store DNA Builder - Stage 5: StoreDNA Builder

## What is Stage 5?

Stage 5 converts the **separate Stage 4 modality vectors** into a single **store-level fingerprint** named `store_dna_vector`.

This implementation uses **weighted late fusion**:

1. Load one vector per store for each modality from Stage 4.
2. L2-normalize each modality matrix row-wise.
3. Apply modality weights.
4. Concatenate weighted modality blocks into one long vector.
5. L2-normalize the final fused vector.

---

## Inputs & outputs

| Input | Source | Stage 5 action | Output |
|-------|--------|----------------|--------|
| `*_vectors.npz` | Stage 4 | Weighted late fusion | `store_dna_vectors.npz` |
| `store_modality_index.csv` | Stage 4 | Coverage / QC | `store_dna_index.csv` |

**Output directory:** `data/USA_100_Stores/store_dna/`

With the default settings, the fused vector dimension is:

- `5 x 3072` embedding dims
- `+ 5` structured ops dims
- `= 15365` dims total


## 1. Setup

In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.store_dna_builder import (
    ALL_MODALITIES,
    DEFAULT_WEIGHTS,
    StoreDNABuilderConfig,
    build_store_dna,
    load_stage4_vectors,
    weighted_late_fusion,
)

MODALITY_DIR = PROJECT_ROOT / "data" / "USA_100_Stores" / "modality_vectors"
STORE_DNA_DIR = PROJECT_ROOT / "data" / "USA_100_Stores" / "store_dna"

print("Project root:", PROJECT_ROOT)
print("Stage 4 input:", MODALITY_DIR)
print("Stage 5 output:", STORE_DNA_DIR)


## 2. Configuration

In [ ]:
MAX_STORES = None
L2_NORMALIZE_INPUTS = True
L2_NORMALIZE_OUTPUT = True
WEIGHTS = DEFAULT_WEIGHTS.copy()

config = StoreDNABuilderConfig(
    max_stores=MAX_STORES,
    l2_normalize_inputs=L2_NORMALIZE_INPUTS,
    l2_normalize_output=L2_NORMALIZE_OUTPUT,
    weights=WEIGHTS,
)

print("Max stores:", config.max_stores or "all")
print("L2 normalize inputs:", config.l2_normalize_inputs)
print("L2 normalize output:", config.l2_normalize_output)
print("Weights:", json.dumps(config.weights, indent=2))


## 3. Preview Stage 4 inputs

In [ ]:
loaded = load_stage4_vectors(MODALITY_DIR)
preview_rows = []
for modality in ALL_MODALITIES:
    store_ids, vectors = loaded[modality]
    preview_rows.append({
        "modality": modality,
        "stores": len(store_ids),
        "vector_dim": vectors.shape[1],
    })
pd.DataFrame(preview_rows)


## 4. Preview the fusion shape

In [ ]:
base_store_ids = loaded["reviews"][0]
aligned = {modality: loaded[modality][1] for modality in ALL_MODALITIES}
preview_fused, preview_meta = weighted_late_fusion(
    aligned,
    config.weights,
    l2_normalize_inputs=config.l2_normalize_inputs,
)
print("Stores:", len(base_store_ids))
print("Preview fused shape:", preview_fused.shape)
pd.DataFrame(preview_meta)


## 5. Run full Stage 5 - build store DNA vectors

In [ ]:
manifest = build_store_dna(MODALITY_DIR, STORE_DNA_DIR, config)
print(json.dumps(manifest, indent=2))


## 6. QC - output coverage

In [ ]:
index_df = pd.read_csv(STORE_DNA_DIR / "store_dna_index.csv")
index_df.head()


In [ ]:
store_dna_npz = np.load(STORE_DNA_DIR / "store_dna_vectors.npz")
store_ids = store_dna_npz["store_ids"]
vectors = store_dna_npz["vectors"]
print(f"store_dna_vectors.npz: {len(store_ids)} stores x {vectors.shape[1]} dims")
print("Average final vector norm:", float(np.linalg.norm(vectors, axis=1).mean()))
print("Non-zero rows:", int((vectors != 0).any(axis=1).sum()))


## 7. Output layout

```
data/USA_100_Stores/store_dna/
├── store_dna_vectors.npz
├── store_dna_index.csv
└── store_dna_manifest.json
```

---

## 8. Next steps - Stage 6 (Vector Index)

| Action | Description |
|--------|-------------|
| Create store index | One document per store |
| Upload vector | Index `store_dna_vector` in Azure AI Search |
| Use cases | Peer stores, clustering, similarity search |
